In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import files

# Prompt user to upload file
print("Please upload your raw data Excel file...")
uploaded = files.upload()

# Get the filename
file_path = list(uploaded.keys())[0]
print(f"\nFile '{file_path}' uploaded successfully!\n")

# Load the raw data
df = pd.read_excel(file_path)

print(f"Loaded {len(df)} rows of data")
print(f"Columns: {df.columns.tolist()}\n")

# Forward-fill CMJ data within each (Date, PitcherID) session
print("Forward-filling CMJ data within sessions...")
cmj_cols = [
    'Concentric Duration [ms]',
    'Concentric Mean Force [N]',
    'Concentric Mean Power / BM [W/kg]',
    'Concentric Peak Force [N]',
    'Eccentric:Concentric Mean Force Ratio [%]',
    'Eccentric Deceleration Phase Duration [s]',
    'Eccentric Duration [ms]',
    'Eccentric Deceleration Mean Force [N]',
    'Eccentric Peak Power [W]',
    'Eccentric Peak Power / BM [W/kg]',
    'Eccentric Peak Velocity [m/s]'
]

df_filled = df.copy()
df_filled[cmj_cols] = df_filled.groupby(['Date', 'PitcherID'])[cmj_cols].ffill()

# Remove rows with any NaN in CMJ or Trackman columns
trackman_cols = [
    'RelSpeed',
    'SpinRate',
    'SpinAxis',
    'VertBreak',
    'InducedVertBreak',
    'HorzBreak'
]

all_analysis_cols = cmj_cols + trackman_cols
df_clean = df_filled.dropna(subset=all_analysis_cols)

print(f"After cleaning: {len(df_clean)} rows\n")

# Calculate fastball percentage per pitcher
pitcher_pitch_counts = df_clean.groupby(['PitcherID', 'TaggedPitchType']).size().reset_index(name='count')
pitcher_totals = df_clean.groupby('PitcherID').size().reset_index(name='total')

fastball_counts = pitcher_pitch_counts[pitcher_pitch_counts['TaggedPitchType'] == 'Fastball'].copy()
fastball_counts = fastball_counts.merge(pitcher_totals, on='PitcherID')
fastball_counts['fb_pct'] = fastball_counts['count'] / fastball_counts['total']

# Classify pitchers
fb_pitchers = fastball_counts[fastball_counts['fb_pct'] >= 0.50]['PitcherID'].values
os_pitchers = fastball_counts[fastball_counts['fb_pct'] < 0.50]['PitcherID'].values

print(f"Fastball-dominant pitchers: {len(fb_pitchers)}")
print(f"Offspeed-dominant pitchers: {len(os_pitchers)}\n")

# Add pitcher type to dataframe
df_clean['PitcherType'] = df_clean['PitcherID'].apply(
    lambda x: 'Fastball' if x in fb_pitchers else 'Offspeed'
)

def create_correlation_matrix(data, title, output_name):
    """Create correlation matrix heatmap"""
    if len(data) < 10:
        print(f"Not enough data for {title} (n={len(data)})")
        return None

    # Select only the analysis columns
    corr_data = data[all_analysis_cols].copy()

    # Calculate correlation matrix
    corr_matrix = corr_data.corr()

    # Create mask for upper triangle
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

    # Create figure
    plt.figure(figsize=(16, 14))

    # Create heatmap
    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        center=0,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8},
        vmin=-1,
        vmax=1,
        annot_kws={'size': 8}
    )

    plt.title(f'{title}\n(n={len(data)} pitches)', fontsize=16, pad=20)
    plt.tight_layout()

    # Save figure
    plt.savefig(f'{output_name}.png', dpi=300, bbox_inches='tight')
    print(f"Saved: {output_name}.png")

    # Also save correlation matrix as Excel
    corr_matrix.to_excel(f'{output_name}.xlsx')
    print(f"Saved: {output_name}.xlsx")

    plt.show()

    return corr_matrix

# Create correlation matrices for each stratification
print("="*80)
print("GENERATING CORRELATION MATRICES")
print("="*80 + "\n")

# Fastball Pitchers - Fastballs
fb_fb = df_clean[(df_clean['PitcherType'] == 'Fastball') &
                  (df_clean['TaggedPitchType'] == 'Fastball')]
create_correlation_matrix(fb_fb,
                          'Fastball Pitchers: Fastball Correlations',
                          'FastballPitchers_Fastball')

# Fastball Pitchers - ChangeUp
fb_ch = df_clean[(df_clean['PitcherType'] == 'Fastball') &
                  (df_clean['TaggedPitchType'] == 'ChangeUp')]
create_correlation_matrix(fb_ch,
                          'Fastball Pitchers: ChangeUp Correlations',
                          'FastballPitchers_ChangeUp')

# Fastball Pitchers - Slider
fb_sl = df_clean[(df_clean['PitcherType'] == 'Fastball') &
                  (df_clean['TaggedPitchType'] == 'Slider')]
create_correlation_matrix(fb_sl,
                          'Fastball Pitchers: Slider Correlations',
                          'FastballPitchers_Slider')

# Fastball Pitchers - Curveball
fb_cb = df_clean[(df_clean['PitcherType'] == 'Fastball') &
                  (df_clean['TaggedPitchType'] == 'Curveball')]
create_correlation_matrix(fb_cb,
                          'Fastball Pitchers: Curveball Correlations',
                          'FastballPitchers_Curveball')

# Offspeed Pitchers - Fastballs
os_fb = df_clean[(df_clean['PitcherType'] == 'Offspeed') &
                  (df_clean['TaggedPitchType'] == 'Fastball')]
create_correlation_matrix(os_fb,
                          'Offspeed Pitchers: Fastball Correlations',
                          'OffspeedPitchers_Fastball')

# Offspeed Pitchers - ChangeUp
os_ch = df_clean[(df_clean['PitcherType'] == 'Offspeed') &
                  (df_clean['TaggedPitchType'] == 'ChangeUp')]
create_correlation_matrix(os_ch,
                          'Offspeed Pitchers: ChangeUp Correlations',
                          'OffspeedPitchers_ChangeUp')

# Offspeed Pitchers - Slider
os_sl = df_clean[(df_clean['PitcherType'] == 'Offspeed') &
                  (df_clean['TaggedPitchType'] == 'Slider')]
create_correlation_matrix(os_sl,
                          'Offspeed Pitchers: Slider Correlations',
                          'OffspeedPitchers_Slider')

# Offspeed Pitchers - Curveball
os_cb = df_clean[(df_clean['PitcherType'] == 'Offspeed') &
                  (df_clean['TaggedPitchType'] == 'Curveball')]
create_correlation_matrix(os_cb,
                          'Offspeed Pitchers: Curveball Correlations',
                          'OffspeedPitchers_Curveball')

print("\n" + "="*80)
print("ALL CORRELATION MATRICES GENERATED!")
print("="*80)
print("\nFiles created:")
print("- 8 PNG heatmap images")
print("- 8 Excel files with correlation values")